In [ ]:
from aqua import Reader

reader = Reader(
    url="https://eerie.cloud.dkrz.de/datasets/HadGEM3-GC5E-HH.historical.v20250409.gr1.Amon/stac",
    loglevel="debug",
    stac_kwargs="eerie-cloud",
    areas=False,
)
data = reader.retrieve()

In [1]:
import pystac

url = "https://eerie.cloud.dkrz.de/datasets/HadGEM3-GC5E-HH.historical.v20250409.gr1.Amon/stac"
stack_kwags = "eerie-cloud"
item = pystac.read_file(url)
obj = item.assets["eerie-cloud"]
obj

<Asset href=https://eerie.cloud.dkrz.de/datasets/HadGEM3-GC5E-HH.historical.v20250409.gr1.Amon/kerchunk>

In [2]:
import xarray as xr

xr.open_dataset(
    obj.href, **obj.extra_fields.get("xarray:open_kwargs", {}), storage_options=obj.extra_fields.get("xarray:storage_options")
)

/opt/homebrew/Caskroom/miniforge/base/envs/aqua/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'clt' has multiple fill values {np.float32(0.0), np.float64(1.0000000200408773e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/opt/homebrew/Caskroom/miniforge/base/envs/aqua/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'pr' has multiple fill values {np.float32(0.0), np.float64(1.0000000200408773e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/opt/homebrew/Caskroom/miniforge/base/envs/aqua/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'sfcWind' has multiple fill values {np.float32(0.0), np.float64(1.0000000200408773e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/opt/homebrew/Caskroom/miniforge/base/envs/aqua/lib/python3.14/site-packages/xarray/conventions.py:205: Serializati

KeyboardInterrupt: 

In [ ]:
import pystac
import xarray as xr


def get_asset(url, stac_kwargs):
    obj = pystac.read_file(url)

    # Follow nested dict: each key is a child or item id
    while isinstance(stac_kwargs, dict):
        ((key, stac_kwargs),) = stac_kwargs.items()  # exactly one entry per level
        nxt = obj.get_child(key) if hasattr(obj, "get_child") else None
        if nxt is None and hasattr(obj, "get_item"):
            nxt = obj.get_item(key)
        if nxt is None:
            raise KeyError(f"'{key}' not found in '{obj.id}'")
        obj = nxt

    # Leaf: a string is the asset key of the current object
    if stac_kwargs not in obj.assets:
        raise KeyError(f"asset '{stac_kwargs}' not found in '{obj.id}'. Available: {list(obj.assets)}")
    return obj.assets[stac_kwargs]


def open_asset(asset):
    kw = dict(asset.extra_fields.get("xarray:open_kwargs", {}))
    so = asset.extra_fields.get("xarray:storage_options")
    if so:
        kw.setdefault("storage_options", so)
    return xr.open_dataset(asset.href, **kw)

In [ ]:
asset = get_asset("https://eerie.cloud.dkrz.de/datasets/HadGEM3-GC5E-HH.historical.v20250409.gr1.Amon/stac", "eerie-cloud")
asset

In [ ]:
asset = get_asset(
    "https://km-scale-cloud.dkrz.de/stac-catalog-all.json",
    {"km-scale-cloud-dkrz-era5-collection": {"era5-dkrz.surface_analysis_monthly": "eerie-cloud"}},
)
asset

In [ ]:
url = "https://km-scale-cloud.dkrz.de/stac-catalog-all.json"
item = "era5-dkrz.surface_analysis_monthly"
asset = "eerie-cloud"
root = pystac.read_file(url)
item = root.get_item(asset, recursive=True)
asset = item.assets[asset]
asset